## Simulator (fleet of uavs)

In [1]:
from simulator import Simulator
from simulator.config import Color
from simulator.entities import SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import (
    QGC,
    Gazebo,
    GazMarker,
    NoVisualizer,
    QGCMarker,
)

clean()

## Simulation Positions

In [2]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241,alt=0,heading=0) 
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading) 


base_homes= ENUPose.list([(50, 0, 0, 0),(-25, 43.3, 0, 0),(-25, -43.3, 0, 0)])
base_paths = [ENU.list([(0, 0, 5),(-100, 0, 5)]),
              ENU.list([(0, 0, 5),(50, -86.6, 5)]),
              ENU.list([(0, 0, 5),(50, 86.6, 5)])]

## Create Vehicles

In [3]:
sysids = [1,2,3]
colors=[Color.BLUE,Color.RED,Color.YELLOW]

speeds=[2.0,2.0,2.0]  # m/s
lands=[True,True,True]
gcs_name =  f'Multicolor_{"".join([color.emoji for color in colors])}'

vehs:list[SimVehicle] = []
for sysid, base_home, base_path, color,speed,land in zip(
    sysids, base_homes, base_paths,colors,speeds,lands):
    auto_plan= AutoPlan.from_relative_path(name="simple_auto_plan",
                                           sysid=sysid,
                                           gra_origin=gra_origin,
                                           relative_home=base_home,
                                           relative_path=base_path,
                                           land=land,
                                           navigation_speed=speed)

    veh = SimVehicle.from_relative(
        sysid=sysid,
        gcs_name=gcs_name,
        plan=auto_plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=base_home,
        relative_path=base_path,
    )
    vehs.append(veh)

## Visualizer

### Gazebo

In [4]:
gaz= Gazebo(gra_origin,world_path="simulator/visualizer/gazebo/worlds/runway5.world")
origin_gaz = GazMarker(name="origin",
                    group="origin",
                    pos=enu_origin.unpose(),
                    color=Color.WHITE)
gaz.markers.append(origin_gaz)

### QGroundControl

In [5]:
qgc= QGC(gra_origin)
origin_qgc = QGCMarker(name="origin",
                pos=gra_origin.unpose(),
                color=Color.WHITE
                )
qgc.markers.append(origin_qgc)

### No Visualizer

In [6]:
novis = NoVisualizer(gra_origin)

## Simulator

In [7]:
simulator = Simulator(
	visualizer=qgc,
	terminals=['gcs'],
	verbose=1,
)

for veh in vehs:
    simulator.add_vehicle(veh)

simulator.show()

/home/abeldg/uav-cyber-sim/simulator/helpers/coordinates.py:412: UserWarning: color argument of Icon should be one of: {'beige', 'lightgreen', 'white', 'darkblue', 'lightred', 'red', 'cadetblue', 'pink', 'orange', 'gray', 'darkpurple', 'darkgreen', 'green', 'lightgray', 'blue', 'black', 'darkred', 'purple', 'lightblue'}.
  location=[self.lat, self.lon], popup=label, icon=folium.Icon(color=color)


In [ ]:
orac = simulator.launch()
orac.run()

13:07:09 - Oracle ⚪ - INFO - 🗺️  QGroundControl launched for 2D visualization — simulation powered by ArduPilot SITL.
13:07:09 - Oracle ⚪ - INFO - 🚀 GCS Multicolor_🟦🟥🟨 launched (PID 2772263)
13:07:09 - Oracle ⚪ - INFO - 🏁 Starting Oracle with 3 vehicles and 1 GCSs


## Run